# Random Matrix Theory: Marchenko-Pastur and Tracy-Widom

Verifies Jacobian eigenvalue statistics match RMT predictions. Used for H1 KS-test.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Marchenko-Pastur Law

In [ ]:
from src.core.spectral.marchenko_pastur import MarchenkoPasturDistribution

n, m = 64, 256
beta = n / m  # aspect ratio
mp = MarchenkoPasturDistribution(beta=beta, sigma2=1.0)
print(f"Support: [{mp.lam_minus:.3f}, {mp.lam_plus:.3f}]")
print(f"Mean: {mp.mean:.3f}, Variance: {mp.variance:.3f}")

# Sample Wishart eigenvalues
rng = np.random.default_rng(42)
ev_sample = mp.sample_wishart(n, m, rng=rng)
ks_stat, p_val = mp.ks_test(ev_sample)
print(f"KS test: stat={ks_stat:.4f}, p={p_val:.4f}")
assert p_val > 0.05, f"KS test failed: p={p_val:.4f}"

# Plot
lam_grid = np.linspace(mp.lam_minus * 0.5, mp.lam_plus * 1.5, 300)
plt.figure(figsize=(7,4))
plt.hist(ev_sample, bins=40, density=True, alpha=0.6, label="Wishart eigenvalues")
plt.plot(lam_grid, mp.pdf(lam_grid), "r-", lw=2, label="Marchenko-Pastur PDF")
plt.xlabel("λ"); plt.ylabel("density"); plt.legend()
plt.title(f"Marchenko-Pastur Law (β={beta:.2f})")
plt.tight_layout(); plt.show()
